# Customer Segmentation for Marketing Strategy

## Executive Summary

## 1.) Business Problem

### Business Questions

- What distinct customer segments can be identified based on customer characteristics and purchasing behavior?
- How do these segments differ in customer value, spending, and engagement?
- Which customer segments should be prioritized for targeted marketing?
- How can marketing strategies be tailored to the characteristics and needs of each segment?

## 2.) Setup and Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

## 3.) Data Overview

### 3.1 Load and Preview Dataset

Load the dataset and preview the first five records to understand the available variables and the overall structure of the data.

In [2]:
df = pd.read_csv('../data/marketing_campaign.csv', sep='\t')

df.head()

,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.0,0,0,04-09-2012,58,635,...,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.0,1,1,08-03-2014,38,11,...,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.0,0,0,21-08-2013,26,426,...,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,26646.0,1,0,10-02-2014,26,11,...,6,0,0,0,0,0,0,3,11,0
4,5324,1981,PhD,Married,58293.0,1,0,19-01-2014,94,173,...,5,0,0,0,0,0,0,3,11,0


### 3.2 Data Structure

Inspect the number of records, column names, data types, and non-null counts.

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2240 entries, 0 to 2239
Data columns (total 29 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID                   2240 non-null   int64  
 1   Year_Birth           2240 non-null   int64  
 2   Education            2240 non-null   str    
 3   Marital_Status       2240 non-null   str    
 4   Income               2216 non-null   float64
 5   Kidhome              2240 non-null   int64  
 6   Teenhome             2240 non-null   int64  
 7   Dt_Customer          2240 non-null   str    
 8   Recency              2240 non-null   int64  
 9   MntWines             2240 non-null   int64  
 10  MntFruits            2240 non-null   int64  
 11  MntMeatProducts      2240 non-null   int64  
 12  MntFishProducts      2240 non-null   int64  
 13  MntSweetProducts     2240 non-null   int64  
 14  MntGoldProds         2240 non-null   int64  
 15  NumDealsPurchases    2240 non-null   int64  
 16 

### 3.3 Summary Statistics

Review summary statistics for numerical variables to identify their ranges, distributions, and potential anomalies.

In [4]:
df.describe()

,ID,Year_Birth,Income,Kidhome,Teenhome,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
count,2240.000000,2240.000000,2216.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,...,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.0,2240.0,2240.000000
mean,5592.159821,1968.805804,52247.251354,0.444196,0.506250,49.109375,303.935714,26.302232,166.950000,37.525446,...,5.316518,0.072768,0.074554,0.072768,0.064286,0.013393,0.009375,3.0,11.0,0.149107
std,3246.662198,11.984069,25173.076661,0.538398,0.544538,28.962453,336.597393,39.773434,225.715373,54.628979,...,2.426645,0.259813,0.262728,0.259813,0.245316,0.114976,0.096391,0.0,0.0,0.356274
min,0.000000,1893.000000,1730.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.0,11.0,0.000000
25%,2828.250000,1959.000000,35303.000000,0.000000,0.000000,24.000000,23.750000,1.000000,16.000000,3.000000,...,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.0,11.0,0.000000
50%,5458.500000,1970.000000,51381.500000,0.000000,0.000000,49.000000,173.500000,8.000000,67.000000,12.000000,...,6.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.0,11.0,0.000000
75%,8427.750000,1977.000000,68522.000000,1.000000,1.000000,74.000000,504.250000,33.000000,232.000000,50.000000,...,7.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.0,11.0,0.000000
max,11191.000000,1996.000000,666666.000000,2.000000,2.000000,99.000000,1493.000000,199.000000,1725.000000,259.000000,...,20.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,3.0,11.0,1.000000


### 3.4 Missing Values

Check for missing values across all variables before data cleaning.

In [5]:
df.isnull().sum().sort_values(ascending=False)

Income                 24
ID                      0
Year_Birth              0
Education               0
Marital_Status          0
Kidhome                 0
Teenhome                0
Dt_Customer             0
Recency                 0
MntWines                0
MntFruits               0
MntMeatProducts         0
MntFishProducts         0
MntSweetProducts        0
MntGoldProds            0
NumDealsPurchases       0
NumWebPurchases         0
NumCatalogPurchases     0
NumStorePurchases       0
NumWebVisitsMonth       0
AcceptedCmp3            0
AcceptedCmp4            0
AcceptedCmp5            0
AcceptedCmp1            0
AcceptedCmp2            0
Complain                0
Z_CostContact           0
Z_Revenue               0
Response                0
dtype: int64

### 3.5 Duplicate Records

Check for duplicate customer records that could affect subsequent analysis.

In [6]:
df.duplicated().sum()

np.int64(0)

### 3.6 Categorical Value Counts

Review categorical variables to identify their distinct values and detect potentially inconsistent or low-frequency categories.

In [7]:
for col in ['Education', 'Marital_Status']:
    print(f"\n{col}")
    print(df[col].value_counts())


Education
Education
Graduation    1127
PhD            486
Master         370
2n Cycle       203
Basic           54
Name: count, dtype: int64

Marital_Status
Marital_Status
Married     864
Together    580
Single      480
Divorced    232
Widow        77
Alone         3
Absurd        2
YOLO          2
Name: count, dtype: int64


## 4.) Data Cleaning

### 4.1 Inspect Missing `Income`

The `Income` column contains 24 missing values. Inspect the affected records to understand whether the missing values can be reasonably addressed before customer segmentation.

In [8]:
df[df['Income'].isnull()]

,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
10,1994,1983,Graduation,Married,NaN,1,0,15-11-2013,11,5,...,7,0,0,0,0,0,0,3,11,0
27,5255,1986,Graduation,Single,NaN,1,0,20-02-2013,19,5,...,1,0,0,0,0,0,0,3,11,0
43,7281,1959,PhD,Single,NaN,0,0,05-11-2013,80,81,...,2,0,0,0,0,0,0,3,11,0
48,7244,1951,Graduation,Single,NaN,2,1,01-01-2014,96,48,...,6,0,0,0,0,0,0,3,11,0
58,8557,1982,Graduation,Single,NaN,1,0,17-06-2013,57,11,...,6,0,0,0,0,0,0,3,11,0
71,10629,1973,2n Cycle,Married,NaN,1,0,14-09-2012,25,25,...,8,0,0,0,0,0,0,3,11,0
90,8996,1957,PhD,Married,NaN,2,1,19-11-2012,4,230,...,9,0,0,0,0,0,0,3,11,0
91,9235,1957,Graduation,Single,NaN,1,1,27-05-2014,45,7,...,7,0,0,0,0,0,0,3,11,0
92,5798,1973,Master,Together,NaN,0,0,23-11-2013,87,445,...,1,0,0,0,0,0,0,3,11,0
128,8268,1961,PhD,Married,NaN,0,1,11-07-2013,23,352,...,6,0,0,0,0,0,0,3,11,0


**Observation:** The 24 records with missing `Income` contain complete customer and purchasing information but lack the income value required for customer profiling.

### 4.2 Handle Missing `Income`

Because only 24 out of 2,240 records (approximately 1.07% of the dataset) have missing `Income`, the affected records will be removed rather than imputing income values that could introduce assumptions into the segmentation.

In [9]:
df = df.dropna(subset=['Income'])

print(f'Remaining rows: {df.shape[0]:,}')

Remaining rows: 2,216


**Observation:** The 24 records with missing `Income` were removed, leaving 2,216 customers for further analysis.

### 4.3 Identify Potential `Year_Birth` Outliers

The `Year_Birth` variable contains unusually low values based on the summary statistics. To systematically identify potential outliers without selecting an arbitrary year threshold, the Interquartile Range (IQR) method is used as an initial screening rule.

In [10]:
Q1_birth = df['Year_Birth'].quantile(0.25)
Q3_birth = df['Year_Birth'].quantile(0.75)
IQR_birth = Q3_birth - Q1_birth

lower_bound_birth = Q1_birth - 1.5 * IQR_birth

print(f'Q1: {Q1_birth:.0f}')
print(f'Q3: {Q3_birth:.0f}')
print(f'IQR: {IQR_birth:.0f}')
print(f'Lower bound: {lower_bound_birth:.0f}')

Q1: 1959
Q3: 1977
IQR: 18
Lower bound: 1932


**Observation:** The lower IQR bound for `Year_Birth` is 1932. Values below this threshold are flagged as potential outliers for further inspection.

### 4.4 Investigate Potential `Year_Birth` Outliers

Inspect the records identified by the IQR screening rule to determine whether the values are plausible customer birth years or data anomalies.

In [11]:
df[df['Year_Birth'] < lower_bound_birth][['ID', 'Year_Birth']]

,ID,Year_Birth
192,7829,1900
239,11004,1893
339,1150,1899


**Observation:** Three records have `Year_Birth` values below the IQR lower bound: 1893, 1899, and 1900. Given the customer context, these values imply implausibly high ages and are treated as data anomalies.

### 4.5 Handle `Year_Birth` Anomalies

The three records identified below the IQR lower bound have implausibly high customer ages and will be removed from the analytical dataset.

In [12]:
df = df[df['Year_Birth'] >= lower_bound_birth]

print(f'Remaining rows: {df.shape[0]:,}')

Remaining rows: 2,213


**Observation:** The three records with implausibly low birth years were removed, leaving 2,213 customers for further analysis.

### 4.6 Identify Potential `Income` Outliers

The `Income` variable contains a highly extreme maximum value. The IQR method is used to identify unusually high income values for further inspection rather than automatically removing all high-income customers.

In [13]:
Q1_income = df['Income'].quantile(0.25)
Q3_income = df['Income'].quantile(0.75)
IQR_income = Q3_income - Q1_income

upper_bound_income = Q3_income + 1.5 * IQR_income

print(f'Q1: {Q1_income:,.0f}')
print(f'Q3: {Q3_income:,.0f}')
print(f'IQR: {IQR_income:,.0f}')
print(f'Upper bound: {upper_bound_income:,.0f}')

Q1: 35,246
Q3: 68,487
IQR: 33,241
Upper bound: 118,348


**Observation:** The upper IQR bound for `Income` is approximately 118,350. Values above this threshold are flagged as potential outliers for further inspection.

### 4.7 Investigate Potential `Income` Outliers

Inspect the high-income records identified by the IQR screening rule to distinguish legitimate high-income customers from potential data-entry anomalies.

In [14]:
df[df['Income'] > upper_bound_income][['ID', 'Income']]

,ID,Income
164,8475,157243.0
617,1503,162397.0
655,5555,153924.0
687,1501,160803.0
1300,5336,157733.0
1653,4931,157146.0
2132,11181,156924.0
2233,9432,666666.0


**Observation:** Eight records exceed the IQR upper bound. Seven customers have incomes between approximately 154,000 and 162,000, which may represent legitimate high-income customers. One record has an income of 666,666, making it substantially more extreme than the other potential outliers and warranting treatment as a likely data-entry anomaly.

### 4.8 Handle `Income` Anomaly

The record with an `Income` value of 666,666 is substantially more extreme than the other potential high-income observations and is treated as a likely data-entry anomaly.

In [15]:
df = df[df['Income'] != 666666]

print(f'Remaining rows: {df.shape[0]:,}')

Remaining rows: 2,212


**Observation:** The anomalous income record was removed, leaving 2,212 customers for further analysis.

### 4.9 Review `Marital_Status`

Review the frequency of `Marital_Status` categories to determine whether very low-frequency categories should be consolidated before customer segmentation.

In [16]:
df['Marital_Status'].value_counts()

Marital_Status
Married     857
Together    571
Single      470
Divorced    231
Widow        76
Alone         3
Absurd        2
YOLO          2
Name: count, dtype: int64

In [17]:
df['Marital_Status'] = df['Marital_Status'].replace(
    ['Alone', 'Absurd', 'YOLO'],
    'Other'
)

**Observation:** Most customers belong to the main marital status categories, while `Alone`, `Absurd`, and `YOLO` occur only seven times in total. These very low-frequency categories provide limited representation and are consolidated into `Other` to avoid unnecessary fragmentation in the categorical variable.

### 4.10 Remove Non-Analytical Variables

Remove identifier and constant variables that do not provide meaningful variation for customer segmentation.

In [18]:
df = df.drop(columns=['ID', 'Z_CostContact', 'Z_Revenue'])

print(f'Remaining columns: {df.shape[1]}')

Remaining columns: 26


**Observation:** `ID` was removed because it is an identifier rather than a customer characteristic. `Z_CostContact` and `Z_Revenue` were removed because they contain constant values and therefore provide no variation for segmentation.

### 4.11 Investigate Identical Analytical Profiles

After removing the customer identifier and constant variables, some customers may share identical values across all remaining analytical variables. Inspect these records to determine whether they represent duplicate records or distinct customers with identical observable profiles.

In [19]:
duplicate_profiles = df[df.duplicated(keep=False)].sort_values(
    by=df.columns.tolist()
)

print(f'Rows belonging to identical profiles: {duplicate_profiles.shape[0]:,}')
print(f'Duplicate rows beyond the first occurrence: {df.duplicated().sum():,}')

Rows belonging to identical profiles: 358
Duplicate rows beyond the first occurrence: 182


**Observation:** A total of 358 customer records belong to groups with identical analytical profiles. These groups contain 182 additional repeated rows beyond the first occurrence. The records are not automatically treated as duplicate customers because they may represent distinct customers with identical observable characteristics.

### 4.12 Convert `Dt_Customer`

Convert `Dt_Customer` from text to a datetime variable so that customer tenure and other time-based features can be derived in the feature preparation stage.

In [20]:
df['Dt_Customer'] = pd.to_datetime(
    df['Dt_Customer'],
    format='%d-%m-%Y'
)

print(df['Dt_Customer'].dtype)

datetime64[us]


**Observation:** `Dt_Customer` was successfully converted to a datetime variable and is ready for time-based feature engineering.

### 4.13 Final Data Quality Check

Verify that the cleaned dataset contains no remaining missing values and confirm the final number of records and analytical variables. Identical analytical profiles are also reported but retained because they may represent distinct customers with the same observable characteristics.

In [21]:
print(f'Remaining rows : {df.shape[0]:,}')
print(f'Remaining columns : {df.shape[1]}')
print(f'Total missing : {df.isnull().sum().sum()}')
print(f'Identical analytical profiles : {df.duplicated().sum():,}')

Remaining rows : 2,212
Remaining columns : 26
Total missing : 0
Identical analytical profiles : 182


**Observation:** The cleaned dataset contains 2,212 customers and 26 analytical variables with no missing values. A total of 182 rows are identical across the analytical variables, but these records are retained because they may represent distinct customers with identical observable characteristics.

### 4.14 Cleaning Summary

The following cleaning steps were performed:

- Removed 24 records with missing `Income` values.
- Used the IQR method to identify potential `Year_Birth` outliers.
- Investigated and removed 3 records with implausibly low birth years.
- Used the IQR method to identify potential `Income` outliers.
- Investigated the high-income records and removed 1 likely data-entry anomaly (`Income = 666,666`).
- Consolidated the low-frequency `Marital_Status` categories `Alone`, `Absurd`, and `YOLO` into `Other`.
- Removed `ID`, `Z_CostContact`, and `Z_Revenue` because they do not provide useful variation for customer segmentation.
- Converted `Dt_Customer` from text to datetime for subsequent time-based feature engineering.
- Identified 182 repeated analytical profiles after removing the customer identifier and confirmed that these records may represent distinct customers with identical observable characteristics, so they were retained.

The cleaned dataset contains 2,212 customers and 26 analytical variables and is ready for feature preparation.

## 5.) Feature Assessment & Selection

In [22]:
df.columns.tolist()

['Year_Birth',
 'Education',
 'Marital_Status',
 'Income',
 'Kidhome',
 'Teenhome',
 'Dt_Customer',
 'Recency',
 'MntWines',
 'MntFruits',
 'MntMeatProducts',
 'MntFishProducts',
 'MntSweetProducts',
 'MntGoldProds',
 'NumDealsPurchases',
 'NumWebPurchases',
 'NumCatalogPurchases',
 'NumStorePurchases',
 'NumWebVisitsMonth',
 'AcceptedCmp3',
 'AcceptedCmp4',
 'AcceptedCmp5',
 'AcceptedCmp1',
 'AcceptedCmp2',
 'Complain',
 'Response']

### 5.1 Define the Role of Each Variable

Before selecting clustering features, classify the remaining variables according to the type of information they provide. This helps distinguish variables that describe customer profiles and behavior from variables that represent previous campaign outcomes or administrative information.

In [23]:
variable_roles = {
    'Customer characteristics': [
        'Year_Birth',
        'Education',
        'Marital_Status',
        'Income',
        'Kidhome',
        'Teenhome'
    ],
    
    'Customer relationship': [
        'Dt_Customer',
        'Recency'
    ],
    
    'Purchasing behavior': [
        'MntWines',
        'MntFruits',
        'MntMeatProducts',
        'MntFishProducts',
        'MntSweetProducts',
        'MntGoldProds',
        'NumDealsPurchases',
        'NumWebPurchases',
        'NumCatalogPurchases',
        'NumStorePurchases',
        'NumWebVisitsMonth'
    ],
    
    'Campaign response': [
        'AcceptedCmp1',
        'AcceptedCmp2',
        'AcceptedCmp3',
        'AcceptedCmp4',
        'AcceptedCmp5',
        'Response'
    ],
    
    'Customer experience': [
        'Complain'
    ]
}

for role, variables in variable_roles.items():
    print(f'\n{role}:')
    print(', '.join(variables))


Customer characteristics:
Year_Birth, Education, Marital_Status, Income, Kidhome, Teenhome

Customer relationship:
Dt_Customer, Recency

Purchasing behavior:
MntWines, MntFruits, MntMeatProducts, MntFishProducts, MntSweetProducts, MntGoldProds, NumDealsPurchases, NumWebPurchases, NumCatalogPurchases, NumStorePurchases, NumWebVisitsMonth

Campaign response:
AcceptedCmp1, AcceptedCmp2, AcceptedCmp3, AcceptedCmp4, AcceptedCmp5, Response

Customer experience:
Complain


### 5.2 Decide Which Variables Should Define the Segments

The segmentation should describe differences in customer characteristics, purchasing behavior, and customer relationship rather than differences in previous campaign outcomes. Therefore, variables are evaluated based on whether they describe the customer itself or an outcome of previous marketing activity.

In [24]:
feature_decisions = {
    'Customer characteristics': 'Include',
    'Customer relationship': 'Include after feature transformation',
    'Purchasing behavior': 'Include',
    'Campaign response': 'Exclude from clustering',
    'Customer experience': 'To be evaluated'
}

for category, decision in feature_decisions.items():
    print(f'{category}: {decision}')

Customer characteristics: Include
Customer relationship: Include after feature transformation
Purchasing behavior: Include
Campaign response: Exclude from clustering
Customer experience: To be evaluated


**Observation:** Customer characteristics, relationship variables, and purchasing behavior directly describe customer profiles and are therefore considered for segmentation. Campaign response variables describe outcomes of previous marketing campaigns and are excluded from the clustering feature set so that the segments are not defined by prior campaign responses.

### 5.3 Investigate `Complain`

`Complain` describes whether a customer has previously filed a complaint. Before deciding whether to include it in the clustering feature set, inspect its distribution to determine whether it provides enough variation across customers to meaningfully contribute to segmentation.

In [25]:
df['Complain'].value_counts()

Complain
0    2192
1      20
Name: count, dtype: int64

**Observation:** `Complain` is highly imbalanced: 2,192 customers (99.1%) have no recorded complaint, while only 20 customers (0.9%) have a complaint. The variable has limited variation and may have little influence on broad customer segmentation, so its usefulness as a clustering feature should be evaluated before making the final inclusion decision.

### 5.4 Investigate Customer Profiles by `Complain`

Because `Complain` is highly imbalanced, compare key customer and purchasing metrics between customers with and without recorded complaints. This helps determine whether the small group of customers with complaints represents a materially different customer profile.

The comparison uses representative variables from different dimensions of the customer profile: `Income` represents economic characteristics, `Recency` represents customer activity, selected product spending variables represent purchasing behavior, and `NumWebPurchases` and `NumStorePurchases` represent purchasing channels. The purpose of this comparison is to assess whether customers with complaints show materially different patterns across these dimensions, rather than to select the final clustering features.

In [26]:
df.groupby('Complain')[
    [
        'Income',
        'Recency',
        'MntWines',
        'MntMeatProducts',
        'MntGoldProds',
        'NumWebPurchases',
        'NumStorePurchases'
    ]
].mean().round(2)

,Income,Recency,MntWines,MntMeatProducts,MntGoldProds,NumWebPurchases,NumStorePurchases
Complain,,,,,,,
0,52016.17,49.00,306.46,167.48,44.07,4.09,5.81
1,45672.40,50.75,176.70,117.70,27.60,3.70,5.40


**Observation:** Customers with a recorded complaint represent only 20 of 2,212 customers (0.9%). The complaint group shows lower average income and lower spending across the selected product categories, while average recency and purchase frequency across web and store channels are relatively similar. Although some differences are observed, the small size of the complaint group makes it less suitable as a primary variable for defining broad customer segments.

### 5.5 Decision on `Complain`

`Complain` will be excluded from the clustering feature set because it is highly imbalanced and represents a very small customer subgroup. Although some differences are observed between customers with and without complaints, the variable is more suitable for post-segmentation profiling than for defining the main customer segments. The original variable will therefore be retained in the dataset.

In [27]:
excluded_from_clustering = [
    'AcceptedCmp1',
    'AcceptedCmp2',
    'AcceptedCmp3',
    'AcceptedCmp4',
    'AcceptedCmp5',
    'Response',
    'Complain'
]

print('Variables excluded from clustering:')
for col in excluded_from_clustering:
    print(f'- {col}')

Variables excluded from clustering:
- AcceptedCmp1
- AcceptedCmp2
- AcceptedCmp3
- AcceptedCmp4
- AcceptedCmp5
- Response
- Complain


**Observation:** Campaign response variables and `Complain` are excluded from defining the customer segments, while remaining available in the dataset for post-segmentation profiling. This keeps the clustering focused on customer characteristics, relationship, and purchasing behavior rather than prior campaign outcomes or a very rare complaint indicator.

## 6.) Feature Engineering

The selected variables identified in the previous section represent the main dimensions of customer segmentation. Before preparing these variables for clustering, their representations should be evaluated to determine whether additional transformations or derived features are needed to better capture customer characteristics and behavior.

Any engineered feature will be introduced only when there is a clear analytical rationale and its contribution to the segmentation objective can be explained.

### 6.1 Transform `Year_Birth` into Customer Age

`Year_Birth` represents the customer's year of birth rather than age directly. Age is easier to interpret as a customer characteristic and is more suitable for describing differences between customer segments.

Because the dataset does not provide a common observation date for all customers, age will be estimated using 2014, the latest calendar year represented in the customer registration data. This provides a fixed and reproducible reference year, while the resulting age should be interpreted as an approximation because only the birth year is available.

In [28]:
reference_date = df['Dt_Customer'].max()
reference_year = reference_date.year

df['Age'] = reference_year - df['Year_Birth']

print(f'Reference date : {reference_date.date()}')
print(f'Reference year : {reference_year}')
print(f'Minimum age    : {df["Age"].min()}')
print(f'Median age     : {df["Age"].median():.0f}')
print(f'Maximum age    : {df["Age"].max()}')

Reference date : 2014-06-29
Reference year : 2014
Minimum age    : 18
Median age     : 44
Maximum age    : 74


**Observation:** The customer registration data extends through 2014, which provides a fixed reference year for the age calculation. The resulting estimated age ranges from 18 to 74 years, with a median age of 44 years. Because only the birth year is available, the age feature should be interpreted as an approximation rather than an exact age.

**Decision:** `Age` will be used as the customer age representation for subsequent analysis, while the original `Year_Birth` variable will be retained in the dataset but will not be needed as a separate representation when defining the clustering features.

### 6.2 Transform `Dt_Customer` into Customer Tenure

`Dt_Customer` records when each customer joined the company. The raw registration date is not directly useful as a customer characteristic because its calendar value is less meaningful than the length of the customer's relationship with the company.

To represent relationship duration, calculate customer tenure relative to the latest registration date available in the dataset. This provides a consistent reference point and avoids using the current date, which would make the feature change whenever the notebook is rerun.

In [29]:
df['Tenure_Years'] = (
    (reference_date - df['Dt_Customer']).dt.days / 365.25
)

print(f'Minimum tenure : {df["Tenure_Years"].min():.2f} years')
print(f'Median tenure  : {df["Tenure_Years"].median():.2f} years')
print(f'Maximum tenure : {df["Tenure_Years"].max():.2f} years')

Minimum tenure : 0.00 years
Median tenure  : 0.97 years
Maximum tenure : 1.91 years


**Observation:** Customer tenure ranges from 0 to approximately 1.91 years relative to the latest registration date in the dataset, with a median of approximately 0.97 years. This provides a more interpretable representation of customer relationship duration than the original registration date.

**Decision:** `Tenure_Years` will be used to represent customer relationship duration in subsequent feature preparation. The original `Dt_Customer` variable will be retained for reference but will not be used directly as a clustering feature.

### 6.3 Create Household Feature

`Kidhome` and `Teenhome` describe the number of children and teenagers living in the customer's household. These variables represent related aspects of household composition and can be combined into a single `Children` feature to capture the total number of children and teenagers.

The combined feature provides a simpler representation of household composition without assuming information that is not available in the dataset.

In [30]:
df['Children'] = df['Kidhome'] + df['Teenhome']

print(f'Minimum children : {df["Children"].min()}')
print(f'Median children  : {df["Children"].median():.0f}')
print(f'Maximum children : {df["Children"].max()}')

Minimum children : 0
Median children  : 1
Maximum children : 3


**Observation:** The resulting `Children` feature ranges from 0 to 3, with a median of 1. The feature provides a concise representation of household child composition while preserving the original `Kidhome` and `Teenhome` variables for reference.

**Decision:** `Children` will be retained as an engineered household characteristic. `Kidhome` and `Teenhome` will remain available in the dataset, but the final clustering feature set should avoid using all three representations simultaneously because `Children` is directly derived from the two source variables.

### 6.4 Create Customer Value and Purchase Frequency Features

The dataset contains spending across six product categories and purchase counts across three sales channels. These variables describe detailed purchasing behavior, but summary measures can provide two broader dimensions of customer behavior: total monetary value and purchase frequency.

`TotalSpend` will summarize the customer's total recorded spending across product categories.

`TotalPurchases` will summarize purchase frequency across web, catalog, and store channels. `NumDealsPurchases` is not included in this calculation because it describes purchases made with a discount rather than a separate purchasing channel and may overlap with the channel-based purchase counts.

In [31]:
spending_features = [
    'MntWines',
    'MntFruits',
    'MntMeatProducts',
    'MntFishProducts',
    'MntSweetProducts',
    'MntGoldProds'
]

purchase_channel_features = [
    'NumWebPurchases',
    'NumCatalogPurchases',
    'NumStorePurchases'
]

df['TotalSpend'] = df[spending_features].sum(axis=1)

df['TotalPurchases'] = df[purchase_channel_features].sum(axis=1)

print('TotalSpend')
print(df['TotalSpend'].describe().round(2))

print('\nTotalPurchases')
print(df['TotalPurchases'].describe().round(2))

TotalSpend
count    2212.00
mean      607.27
std       602.51
min         5.00
25%        69.00
50%       397.00
75%      1048.00
max      2525.00
Name: TotalSpend, dtype: float64

TotalPurchases
count    2212.00
mean       12.57
std         7.21
min         0.00
25%         6.00
50%        12.00
75%        18.25
max        32.00
Name: TotalPurchases, dtype: float64


**Observation:** `TotalSpend` ranges from 5 to 2,525, with a median of 397, while `TotalPurchases` ranges from 0 to 32, with a median of 12. These features summarize two broad dimensions of purchasing behavior: customer monetary value and purchase frequency.

**Decision:** `TotalSpend` and `TotalPurchases` will be retained as candidate engineered features for the next stage. `TotalSpend` provides a consolidated measure of monetary value, while `TotalPurchases` provides a consolidated measure of purchase frequency across sales channels. The original spending and channel-specific purchase variables will remain available until the final clustering feature set is determined.

### 6.5 Review Engineered Features

Review the engineered features before moving to clustering-specific preparation. The purpose is to verify that each feature has valid values, a clear interpretation, and a reasonable range after transformation.

This descriptive review does not determine whether the engineered features are independent or sufficiently distinct from their source variables. Potential redundancy between engineered and source variables will be evaluated when defining the final clustering feature set.

In [32]:
engineered_features = [
    'Age',
    'Tenure_Years',
    'Children',
    'TotalSpend',
    'TotalPurchases'
]

df[engineered_features].describe().round(2)

,Age,Tenure_Years,Children,TotalSpend,TotalPurchases
count,2212.00,2212.00,2212.00,2212.00,2212.00
mean,45.09,0.97,0.95,607.27,12.57
std,11.70,0.55,0.75,602.51,7.21
min,18.00,0.00,0.00,5.00,0.00
25%,37.00,0.49,0.00,69.00,6.00
50%,44.00,0.97,1.00,397.00,12.00
75%,55.00,1.45,1.00,1048.00,18.25
max,74.00,1.91,3.00,2525.00,32.00


**Observation:** The engineered features have valid values for all 2,212 customers and summarize five customer dimensions: age, relationship duration, household child composition, monetary value, and purchase frequency. However, `TotalSpend` and `TotalPurchases` are derived from existing spending and purchase-channel variables, so the relationship between the engineered and source variables must be considered before defining the final clustering feature set.

### 6.6 Feature Engineering Summary

The following engineered features were created:

- `Age` to provide an interpretable representation of customer age based on `Year_Birth`.
- `Tenure_Years` to represent customer relationship duration based on `Dt_Customer`.
- `Children` to summarize the number of children and teenagers in the household.
- `TotalSpend` to summarize monetary value across product categories.
- `TotalPurchases` to summarize purchase frequency across web, catalog, and store channels.

The original source variables are retained in the dataset because they may remain useful for profiling and for evaluating alternative feature representations. The engineered features will be evaluated alongside the original variables before the final clustering feature matrix is constructed.